# Project 2: Voice Interview Agent

In this notebook we build a voice-based mock interview agent for an AI Agents Engineer role — introducing each voice-specific concept step by step before assembling the full system.

| Section | Concept |
|---|---|
| 1 | VoicePipeline — speech in, agent thinks, speech out |
| 2 | Memory — multi-turn conversation history |
| 3 | Tools — web search and code interpreter in voice context |
| 4 | Handoffs — routing between specialist agents |
| 5 | Full system — custom workflow, TTS tuning, and streaming |


In [ ]:
# !pip install -q openai-agents[voice] python-dotenv numpy sounddevice

In [10]:
from dotenv import load_dotenv

load_dotenv("../../.env")


True

## Setup


In [11]:
import asyncio
import numpy as np
import sounddevice as sd
from typing import AsyncIterator

from agents import Agent, CodeInterpreterTool, FileSearchTool, Runner, WebSearchTool
from agents.extensions.handoff_prompt import prompt_with_handoff_instructions
from agents.voice import (
    AudioInput,
    SingleAgentVoiceWorkflow,
    SingleAgentWorkflowCallbacks,
    StreamedAudioInput,
    TTSModelSettings,
    VoicePipeline,
    VoicePipelineConfig,
    VoiceWorkflowBase,
    VoiceWorkflowHelper,
)

In [4]:
SAMPLE_RATE = 24000
STOP_PHRASES = {"stop", "exit", "quit", "goodbye", "end interview", "that's all"}


def record_audio(duration: int = 5, sample_rate: int = SAMPLE_RATE) -> np.ndarray:
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype="int16")
    sd.wait()
    return audio.flatten()


def play_audio(chunks: list[np.ndarray], sample_rate: int = SAMPLE_RATE) -> None:
    audio = np.concatenate(chunks)
    sd.play(audio, samplerate=sample_rate)
    sd.wait()


async def play_pipeline_result(result) -> None:
    chunks = []
    async for event in result.stream():
        if event.type == "voice_stream_event_audio":
            chunks.append(event.data)
    if chunks:
        play_audio(chunks)


## Section 1: VoicePipeline Basics

`VoicePipeline` is the core abstraction for voice agents. Each turn follows a fixed path:

**Audio in → Speech-to-text → Agent → Text-to-speech → Audio out**

You wrap any `Agent` in a `SingleAgentVoiceWorkflow` and hand it to `VoicePipeline`. The pipeline handles transcription and synthesis automatically — you only write the agent logic.

This is the simplest possible voice agent: one turn, no memory, no tools.


In [6]:
single_turn_agent = Agent(
    name="SingleTurnVoiceAssistant",
    model="gpt-4o-mini",
    instructions="You are a concise voice assistant. Answer in one short sentence.",
)

async def single_turn_voice_demo(duration: int = 5):
    workflow = SingleAgentVoiceWorkflow(single_turn_agent)
    pipeline = VoicePipeline(workflow=workflow)

    audio = record_audio(duration)
    result = await pipeline.run(AudioInput(buffer=audio))
    await play_pipeline_result(result)

# Run this when you want to test one voice turn.
await single_turn_voice_demo(duration=5)

Recording for 5 seconds...


## Section 2: Memory and Multi-Turn

In Section 1, each call to `pipeline.run()` was stateless — the agent had no memory of earlier turns.

`SingleAgentVoiceWorkflow` solves this by storing conversation history internally. As long as you reuse **the same workflow instance**, every new turn appends to the existing message list and the agent sees the full context.

`SingleAgentWorkflowCallbacks` lets you hook into the workflow lifecycle — here we use it to print the live transcription so we can see what the speech-to-text layer heard.


In [7]:
class TranscriptLogger(SingleAgentWorkflowCallbacks):
    def __init__(self):
        self.last_transcription = ""

    def on_run(self, workflow: SingleAgentVoiceWorkflow, transcription: str) -> None:
        self.last_transcription = transcription
        print(f"You said: {transcription}")

memory_agent = Agent(
    name="MemoryVoiceAssistant",
    model="gpt-4o-mini",
    instructions="""You are a concise voice assistant.
Remember earlier turns and answer in no more than two short sentences.""",
)

async def multi_turn_memory_demo(duration: int = 5, max_turns: int = 3):
    callbacks = TranscriptLogger()
    workflow = SingleAgentVoiceWorkflow(memory_agent, callbacks=callbacks)
    pipeline = VoicePipeline(workflow=workflow)

    for turn in range(1, max_turns + 1):
        command = input(f"Turn {turn}: press Enter to speak, or type 'q' to stop: ")
        if command.lower() in {"q", "quit", "exit"}:
            break

        audio = record_audio(duration)
        result = await pipeline.run(AudioInput(buffer=audio))
        await play_pipeline_result(result)

        if any(phrase in callbacks.last_transcription.lower() for phrase in STOP_PHRASES):
            print("Conversation ended by voice command.")
            break

# Ask two related questions to test memory.
await multi_turn_memory_demo(duration=5, max_turns=3)

Recording for 5 seconds...
You said: हलो
Recording for 5 seconds...
You said: होंडा की गाड़ियां


## Section 3: Tools

Tools work the same way in voice agents as in regular agents — you declare them, the model decides when to call them. The voice pipeline passes tool results back into the conversation automatically.

Two hosted OpenAI tools are particularly useful for an interview agent:

- `WebSearchTool` — lets the agent look up current market expectations, role requirements, or interview trends in real time
- `CodeInterpreterTool` — lets the agent calculate scores, aggregate evaluation signals, or run small computations server-side

The agent's spoken responses stay short (under 35 words) — tools handle the heavy lifting behind the scenes.


In [11]:
import time
from openai import OpenAI

client = OpenAI()

# Upload the candidate's resume PDF and index it into a vector store.
# The interviewer agents will use FileSearchTool to look up specific details
# from the resume rather than relying on an inlined text copy.
with open("../../openai_agents_sdk/docs/IshanDuttaResume.pdf", "rb") as f:
    uploaded_resume = client.files.create(file=f, purpose="assistants")

resume_store = client.vector_stores.create(name="Candidate Resume")
client.vector_stores.files.create(
    vector_store_id=resume_store.id,
    file_id=uploaded_resume.id,
)

while True:
    vs_file = client.vector_stores.files.list(vector_store_id=resume_store.id).data[0]
    if vs_file.status == "completed":
        break
    time.sleep(1)

print("Resume vector store ready:", resume_store.id)

job_description = """
AI Agents SDK Engineer — OpenAI

You will join the OpenAI Agents SDK team to design, build, and teach the patterns
developers use to create production-grade AI agents.

Responsibilities:
- Architect multi-agent orchestration patterns: routing, handoffs, and shared context
  using RunContextWrapper and Agent[T] generic typing
- Build and maintain hosted tools: FileSearchTool with vector store indexing,
  WebSearchTool, CodeInterpreterTool, and custom @function_tool integrations
- Implement voice-capable agents using VoicePipeline, SingleAgentVoiceWorkflow,
  VoiceWorkflowBase, and TTSModelSettings
- Enforce output grounding with ModelSettings(tool_choice="required") and
  design structured outputs using Pydantic BaseModel with output_type
- Write developer documentation, tutorials, and workshop notebooks that teach
  agent concepts progressively (basic agent → tools → handoffs → full system)
- Evaluate reliability, latency, and safety of agentic systems before production

Requirements:
- Strong Python with experience designing developer-facing SDKs or APIs
- Hands-on experience building agents with tool calling, handoffs, and context management
- Familiarity with vector stores, embeddings, and document retrieval (FileSearchTool / RAG)
- Experience with voice interfaces, audio pipelines, or real-time streaming
- Understanding of structured outputs, Pydantic models, and multi-agent evaluation

Nice to have:
- Experience teaching or creating developer education content (workshops, notebooks)
- Contributions to open-source agent or ML frameworks
- Background in ML engineering, generative AI, or LLM fine-tuning
"""

official_interview_tools = [
    WebSearchTool(),
    FileSearchTool(vector_store_ids=[resume_store.id]),
    CodeInterpreterTool(
        tool_config={"type": "code_interpreter", "container": {"type": "auto"}},
    ),
]


In [ ]:
tool_interviewer_agent = Agent(
    name="ToolInterviewer",
    model="gpt-4o",
    instructions=f"""You are a mock interviewer conducting a voice interview for the following role:

{job_description}

Use the official hosted tools when useful:
- Use FileSearchTool to look up specific details from the candidate's resume.
- Use WebSearchTool for current OpenAI Agents SDK expectations and interview context.
- Use CodeInterpreterTool for scoring rubrics or aggregating evaluation signals.

Ask one easy and basic focused question at a time based on the role requirements.
Keep every spoken response under 35 words.""",
    tools=official_interview_tools,
)

async def personalized_voice_question(duration: int = 5):
    workflow = SingleAgentVoiceWorkflow(tool_interviewer_agent)
    pipeline = VoicePipeline(workflow=workflow)

    audio = record_audio(duration)
    result = await pipeline.run(AudioInput(buffer=audio))
    await play_pipeline_result(result)

# Say: "Start the AI agents interview."
await personalized_voice_question(duration=5)


Recording for 5 seconds...


## Section 4: Handoffs

A single agent can only do so much. **Handoffs** let you split responsibility across specialist agents and route the conversation between them at runtime.

Each agent declares:
- `handoffs=[...]` — the list of agents it can hand off to
- `handoff_description` — a short label the orchestrator uses to decide when to route there

`SingleAgentVoiceWorkflow` tracks which agent is currently active and switches automatically when a handoff occurs. The candidate never hears the transition — the voice stream continues uninterrupted.

Here the interview flows: `GreeterAgent` → `QuestionerAgent` → `FeedbackAgent`.


In [ ]:
voice_response_rules = f"""
You are speaking aloud. Keep every response under 35 words.
Ask one easy question at a time. Do not use bullet points.

You are interviewing for this role:
{job_description}

Use FileSearchTool to look up the candidate's background and experience from their resume when relevant.
"""

feedback_agent = Agent(
    name="FeedbackAgent",
    handoff_description="Gives the final interview scorecard.",
    model="gpt-4o",
    instructions=voice_response_rules + """
Use CodeInterpreterTool if you need to calculate or aggregate a score.
Create a short final scorecard for the AI Agents Engineer interview.
Mention one strength, one improvement area, and an overall score out of 5.
""",
    tools=official_interview_tools,
)

questioner_agent = Agent(
    name="QuestionerAgent",
    handoff_description="Asks easy AI agents interview questions and evaluates answers.",
    model="gpt-4o",
    instructions=prompt_with_handoff_instructions(voice_response_rules + """
You ask beginner-friendly AI Agents Engineer questions.
Use WebSearchTool if you need current role expectations.
Use CodeInterpreterTool if you need to compute a score.
For each candidate answer, give one short evaluation sentence, then ask the next easy question.
If the candidate wants to stop, hand off to FeedbackAgent.
"""),
    tools=official_interview_tools,
    handoffs=[feedback_agent],
)

greeter_agent = Agent(
    name="GreeterAgent",
    handoff_description="Starts the interview and routes to the questioner.",
    model="gpt-4o",
    instructions=prompt_with_handoff_instructions(voice_response_rules + """
Greet the candidate, explain this will be an easy AI Agents Engineer interview, then hand off to QuestionerAgent.
"""),
    tools=[WebSearchTool()],
    handoffs=[questioner_agent],
)

async def handoff_voice_demo(duration: int = 5, turns: int = 3):
    workflow = SingleAgentVoiceWorkflow(greeter_agent)
    pipeline = VoicePipeline(workflow=workflow)

    for turn in range(1, turns + 1):
        command = input(f"Turn {turn}: press Enter to speak, or type 'q' to stop: ")
        if command.lower() in {"q", "quit", "exit"}:
            break
        audio = record_audio(duration)
        result = await pipeline.run(AudioInput(buffer=audio))
        await play_pipeline_result(result)

# await handoff_voice_demo(duration=5, turns=3)


## Section 5: The Full Interview System

Now we combine everything — memory, tools, handoffs, custom workflow control, and TTS tuning — into a complete voice interview experience.

| Sub-section | What it adds |
|---|---|
| 5.1 Custom Workflow | `VoiceWorkflowBase` subclass: explicit history, stop words, greeting, max questions |
| 5.2 Voice Tuning | `TTSModelSettings` for voice, speed, and delivery style |
| 5.3 Full Interview | Puts it all together: buffered and streaming audio modes |

`VoiceWorkflowBase` gives you full control compared to `SingleAgentVoiceWorkflow` — you implement `on_start()` for the greeting and `run()` for each turn, managing history and agent switching yourself.


In [ ]:
class InterviewVoiceWorkflow(VoiceWorkflowBase):
    def __init__(self, starting_agent: Agent, final_agent: Agent, max_questions: int = 4):
        self.starting_agent = starting_agent
        self.final_agent = final_agent
        self.current_agent = starting_agent
        self.max_questions = max_questions
        self.question_count = 0
        self.history = []
        self.ended = False

    async def on_start(self) -> AsyncIterator[str]:
        yield "Welcome to your AI Agents Engineer mock interview. I will ask easy questions. Say stop when you want to finish."

    async def run(self, transcription: str) -> AsyncIterator[str]:
        print(f"You said: {transcription}")
        self.history.append({"role": "user", "content": transcription})

        should_finish = any(phrase in transcription.lower() for phrase in STOP_PHRASES)
        if should_finish or self.question_count >= self.max_questions:
            self.current_agent = self.final_agent
            self.history.append({"role": "user", "content": "Please end the interview with a short spoken scorecard."})
            self.ended = True
        else:
            self.question_count += 1

        result = Runner.run_streamed(self.current_agent, self.history)
        async for text in VoiceWorkflowHelper.stream_text_from(result):
            yield text

        self.history = result.to_input_list()
        self.current_agent = result.last_agent

workflow_preview = InterviewVoiceWorkflow(greeter_agent, feedback_agent, max_questions=3)


### 5.1 Voice Tuning

`TTSModelSettings` controls how the text-to-speech layer sounds. You can set the voice character, speaking speed, and delivery instructions — the same way a system prompt shapes an agent's text behavior, these settings shape how it speaks.


In [ ]:
interviewer_tts_settings = TTSModelSettings(
    voice="nova",
    speed=1.1,
    instructions="""
Voice: friendly, clear, and professional.
Pacing: concise and slightly brisk.
Delivery: keep each answer under about ten seconds. Pause briefly before each question.
Emotion: supportive interviewer, not dramatic.
""",
)

interview_voice_config = VoicePipelineConfig(
    workflow_name="AI Agents Engineer Voice Interview",
    tts_settings=interviewer_tts_settings,
)


### 5.2 Full Live Interview

Two run modes — choose based on your setup:

- **Buffered** (`run_full_interview`) — press Enter to record each turn, hear the response, repeat. Easier to debug.
- **Streaming** (`run_streamed_interview`) — microphone streams continuously; the pipeline responds as you speak. Closest to a real voice assistant experience.


In [ ]:
async def run_full_interview(duration: int = 8, max_questions: int = 4):
    workflow = InterviewVoiceWorkflow(greeter_agent, feedback_agent, max_questions=max_questions)
    pipeline = VoicePipeline(workflow=workflow, config=interview_voice_config)

    print("AI Agents Engineer voice interview")
    print("First say: start interview")
    print("Then answer each spoken question.")
    print("Say 'stop' in your answer for a spoken final scorecard.")
    print("Type 'q' before a turn to end without final feedback.\n")

    turn = 1
    while not workflow.ended:
        command = input(f"Turn {turn}: press Enter to speak, or type 'q' to stop: ")
        if command.lower() in {"q", "quit", "exit"}:
            break

        audio = record_audio(duration)
        result = await pipeline.run(AudioInput(buffer=audio))
        await play_pipeline_result(result)
        turn += 1

# Main workshop demo.
# await run_full_interview(duration=8, max_questions=4)


In [ ]:
async def stream_microphone_until_enter(streamed_input: StreamedAudioInput, sample_rate: int = SAMPLE_RATE):
    loop = asyncio.get_running_loop()

    def callback(indata, frames, time, status):
        chunk = indata.copy().reshape(-1)
        asyncio.run_coroutine_threadsafe(streamed_input.add_audio(chunk), loop)

    print("Streaming microphone. Speak naturally; press Enter to end the session.")
    with sd.InputStream(samplerate=sample_rate, channels=1, dtype="int16", callback=callback):
        await asyncio.to_thread(input)

    await streamed_input.add_audio(None)

async def run_streamed_interview(max_questions: int = 4):
    workflow = InterviewVoiceWorkflow(greeter_agent, feedback_agent, max_questions=max_questions)
    streamed_input = StreamedAudioInput()
    pipeline = VoicePipeline(workflow=workflow, config=interview_voice_config)

    result = await pipeline.run(streamed_input)
    recorder_task = asyncio.create_task(stream_microphone_until_enter(streamed_input))

    player = sd.OutputStream(samplerate=SAMPLE_RATE, channels=1, dtype=np.int16)
    player.start()

    async for event in result.stream():
        if event.type == "voice_stream_event_audio":
            player.write(event.data)
        elif event.type == "voice_stream_event_lifecycle":
            print(event.event)

    await recorder_task
    player.stop()
    player.close()

# Advanced streaming demo.
# await run_streamed_interview(max_questions=4)
